# OncoEvidence — End-to-End, Inline Reproducible Workflow (Google Colab)

**Mechanism-Guided AI Evidence Triage for Cancer Drug Repurposing**

> **Core question.** For cancer drug repurposing, can a biomedical knowledge graph
> plus a citation-grounded LLM agent prioritize candidates by *both* predicted
> therapeutic relevance *and* mechanistic plausibility — and when does the graph
> add value over a strong tabular baseline?

Unlike a thin "run the scripts" notebook, this one **builds and runs the actual
workflow inline** using the project's own library functions, so you can read,
inspect, and modify every stage in the browser:

- Build the PrimeKG graph and **look inside it** (node/edge types, example
  drugs/diseases/genes, oncology mask, feature tensors).
- **Train the GNN, XGBoost, MLP, and KGE yourself** on a split and watch the
  AUROCs come out; sweep all 3 regimes; run a **topology ablation**.
- **Extract mechanism-of-action paths** for a real indication and inspect them.
- **Compute the true-vs-random separation AUROC** and plot the histogram in-cell.
- **Run the literature verifier** on one path (abstracts + grade + evidence).
- **Train the joint mechanism-recovery GNN** and rank the bridge gene.
- **Produce a repurposing shortlist** as a table.

Each section also keeps a one-line call to the canonical `scripts/*.py` so you can
**reproduce the full published numbers** (`FAST_MODE = False`).

> *All predictions are hypothesis-generating and not medical advice.*

**Recommended:** Runtime → Change runtime type → **GPU** (T4 is enough).

## 0. Configuration switches

In [ ]:
# --- master switches -------------------------------------------------------
FAST_MODE = True   # True: quick inline pass. False: full published reproduction (hours).
RUN_LLM   = False  # True: use the LLM verifier/judge (needs ONCO_LLM_API_KEY below).

# Optional OpenAI-compatible key (e.g. OpenRouter) for the LLM evidence reviewer.
ONCO_LLM_API_KEY  = ""
ONCO_LLM_BASE_URL = "https://openrouter.ai/api/v1"      # OpenAI default: https://api.openai.com/v1
ONCO_LLM_MODEL    = "openai/gpt-4o-mini"

# Inline-training budgets (kept small in FAST_MODE so the notebook finishes quickly;
# the published numbers come from the scripts/*.py calls with FAST_MODE=False).
INLINE_GNN_EPOCHS = 12 if FAST_MODE else 50
INLINE_MLP_EPOCHS = 50 if FAST_MODE else 200
INLINE_KGE_EPOCHS = 80 if FAST_MODE else 300
INLINE_PATIENCE   = 6 if FAST_MODE else 10
INLINE_XGB_TUNE   = (not FAST_MODE)
INLINE_XGB_TRIALS = 5 if FAST_MODE else 8
N_SEP             = 120 if FAST_MODE else 400   # pairs per group for separation AUROC
INLINE_REC_EPOCHS = 12 if FAST_MODE else 60     # mechanism-recovery training epochs

print(f"FAST_MODE={FAST_MODE} | RUN_LLM={RUN_LLM}")
print(f"inline GNN epochs={INLINE_GNN_EPOCHS}, XGB tune={INLINE_XGB_TUNE}, sep pairs/group={N_SEP}")

## 1. Environment setup

Clone the project (on Colab) or locate it, then install dependencies. On Colab,
PyTorch ships pre-installed; we only add the extras (PyG, XGBoost, Optuna, …).

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/akshusuba/research.git"   # OncoEvidence lives in research/oncorepurpose

def find_project_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "oncorepurpose" / "config.py").exists():
            return base
    return None

root = find_project_root()
if root is None:
    if not Path("research").exists():
        print("Cloning repo ...")
        # PRIVATE repo? use: REPO_URL = "https://<TOKEN>@github.com/akshusuba/research.git"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = (Path("research") / "oncorepurpose").resolve()

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["PYTHONPATH"] = str(root)
PY = sys.executable   # robust interpreter for the scripts/*.py reproduction cells
print("Project root:", root)
print("Python:", PY)

In [ ]:
%pip install -q torch-geometric xgboost optuna "sentence-transformers>=2.2.0" "transformers>=4.40,<5" scikit-learn pandas numpy scipy matplotlib seaborn requests tqdm pyyaml
print("Dependencies installed.")

In [ ]:
import torch
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", dev)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Use Runtime > Change runtime type > GPU (inline training is slow on CPU).")

## 2. Data collection — PrimeKG

[PrimeKG](https://github.com/mims-harvard/PrimeKG) is a precision-medicine
knowledge graph. We download the raw `kg.csv` (~980 MB), parse it into a
heterogeneous graph, and attach **shared** node features fed to *both* the GNN
and the XGBoost baseline (so any gap is attributable to topology, not content).

In [ ]:
# 2a. Download PrimeKG (skips if already present; ~980 MB on first run).
from oncorepurpose.data.download import download_primekg
download_primekg()

In [ ]:
# 2b. Parse kg.csv -> PyG HeteroData (cached to data/primekg_hetero.pt).
from oncorepurpose.config import HETERODATA_PT
from oncorepurpose.data.build_graph import build_hetero_from_primekg
if HETERODATA_PT.exists():
    print("HeteroData already built ->", HETERODATA_PT)
else:
    build_hetero_from_primekg()

In [ ]:
# 2c. Load the graph + attach shared node features. FAST_MODE uses deterministic
# hashing features (instant); the full run uses real SentenceTransformer embeddings.
from oncorepurpose.datasets import load_primekg, graph_summary
data, targets = load_primekg(with_features=True, force_fallback_features=FAST_MODE)
target = targets["indication"]
print(graph_summary(data, targets))

In [ ]:
# 2d. Look inside the graph: node types, feature dims, and a few example nodes.
import pandas as pd
from oncorepurpose.config import DRUG_TYPE, DISEASE_TYPE

rows = []
for nt in data.node_types:
    store = data[nt]
    rows.append({
        "node_type": nt,
        "num_nodes": int(store.num_nodes),
        "feature_dim": int(store.x.size(1)) if "x" in store else None,
        "example": ", ".join(str(n) for n in list(getattr(store, "node_names", []))[:2]),
    })
display(pd.DataFrame(rows).sort_values("num_nodes", ascending=False).reset_index(drop=True))

onc = data[DISEASE_TYPE].is_oncology
print(f"\nOncology diseases flagged: {int(onc.sum())} / {int(data[DISEASE_TYPE].num_nodes)}")
print("Target relation:", target, "with", int(data[target].edge_index.size(1)), "indication edges")
print("Edge (relation) types:", len(data.edge_types))

## 3. Aim 1 — Candidate generation: train the models yourself

We rank `drug --indication--> cancer` pairs. Here we **train each model inline** so
you can see exactly how it's done, using the repo's library functions:

- `make_split` builds a leakage-safe split (the message-passing graph contains only
  training edges; all drug↔disease therapeutic edges to held-out nodes are removed).
- `train_gnn` / `train_mlp` / `train_kge` train the encoders; `run_xgboost` trains
  the tuned tabular control on the **same** node features.

**Honest finding:** with a properly tuned XGBoost on rich text features, the GNN
does *not* win on link ranking — a name embedding lets the tabular model take a
semantic-similarity shortcut. This motivates the mechanism-aware sections that follow.

In [ ]:
from oncorepurpose.evaluation.splits import make_split, ablate_topology
from oncorepurpose.evaluation.trainer import (
    set_all_seeds, train_gnn, train_mlp, train_kge, evaluate_model,
)
from oncorepurpose.models import HeteroGNN, FeatureMLP, DistMultKGE
from oncorepurpose.baselines.xgboost_baseline import run_xgboost

in_dims = {nt: int(data[nt].x.size(1)) for nt in data.node_types}
SEED = 0

# 3a. Build a transductive split and inspect it.
split = make_split(data, target, "transductive", seed=SEED)
print("Split info:", split.info)
print("train/val/test labelled pairs:",
      split.train_label.numel(), split.val_label.numel(), split.test_label.numel())

In [ ]:
# 3b. Train all four models on this split and compare (test AUROC/AUPRC/F1).
import pandas as pd

def train_one(name, split):
    set_all_seeds(SEED)
    if name == "GNN":
        m = HeteroGNN(list(data.node_types), list(split.base.edge_types), in_dims,
                      hidden=128, num_layers=2, dropout=0.3)
        m = train_gnn(m, split, dev, epochs=INLINE_GNN_EPOCHS, patience=INLINE_PATIENCE)
        return evaluate_model(m, split, dev)
    if name == "MLP":
        m = FeatureMLP(list(data.node_types), in_dims, hidden=128, dropout=0.3)
        m = train_mlp(m, split, dev, epochs=INLINE_MLP_EPOCHS, patience=INLINE_PATIENCE)
        return evaluate_model(m, split, dev)
    if name == "KGE":
        m = DistMultKGE(DRUG_TYPE, DISEASE_TYPE, int(data[DRUG_TYPE].num_nodes),
                        int(data[DISEASE_TYPE].num_nodes), dim=128)
        m = train_kge(m, split, dev, epochs=INLINE_KGE_EPOCHS, patience=INLINE_PATIENCE)
        return evaluate_model(m, split, dev)
    if name == "XGBoost":
        return run_xgboost(split, data, seed=SEED, n_estimators=400,
                           tune=INLINE_XGB_TUNE, n_trials=INLINE_XGB_TRIALS)

trans_metrics = {}
for name in ["GNN", "XGBoost", "MLP", "KGE"]:
    mt = train_one(name, split)
    trans_metrics[name] = mt
    print(f"  {name:8s} test AUROC={mt['auroc']:.4f}  AUPRC={mt['auprc']:.4f}  F1={mt['f1']:.4f}")

df_trans = pd.DataFrame(trans_metrics).T[["auroc", "auprc", "f1"]].round(4)
display(df_trans)

In [ ]:
# 3c. Sweep all three regimes (one seed inline) and build the headline table.
import matplotlib.pyplot as plt
import numpy as np

REGIMES = [("transductive", {}),
           ("inductive_cold_dst", {"restrict_oncology": True}),
           ("inductive_cold_src", {})]
REGIME_LABEL = {"transductive": "Transductive",
                "inductive_cold_dst": "Inductive cold-disease (onc)",
                "inductive_cold_src": "Inductive cold-drug"}
MODELS = ["GNN", "XGBoost", "MLP", "KGE"]

grid = {}  # regime -> model -> auroc
grid["transductive"] = {m: trans_metrics[m]["auroc"] for m in MODELS}  # reuse 3b

for regime, kw in REGIMES:
    if regime == "transductive":
        continue
    grid[regime] = {}
    sp = make_split(data, target, regime, seed=SEED, **kw)
    print(f"[{regime}] {sp.info}")
    for name in MODELS:
        mt = train_one(name, sp)
        grid[regime][name] = mt["auroc"]
        print(f"  {name:8s} test AUROC={mt['auroc']:.4f}")

df_grid = pd.DataFrame({REGIME_LABEL[r]: grid[r] for r, _ in REGIMES}).T[MODELS].round(3)
print("\nTest AUROC (one seed, inline budgets):")
display(df_grid)

ax = df_grid.plot(kind="bar", figsize=(10, 5), rot=10)
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_ylabel("Test AUROC"); ax.set_ylim(0.4, 1.0)
ax.set_title("GNN vs tabular / memorization baselines (one-seed inline run)")
plt.tight_layout(); plt.show()

In [ ]:
# 3d. Topology ablation: does the GRAPH drive the GNN? Train the GNN on the
# cold-disease split with the topology intact, edge-shuffled, and emptied.
abl = {}
base_split = make_split(data, target, "inductive_cold_dst", seed=SEED, restrict_oncology=True)
for mode in ["intact", "shuffle", "empty"]:
    s = base_split if mode == "intact" else ablate_topology(base_split, mode, seed=SEED)
    set_all_seeds(SEED)
    m = HeteroGNN(list(data.node_types), list(s.base.edge_types), in_dims,
                  hidden=128, num_layers=2, dropout=0.3)
    m = train_gnn(m, s, dev, epochs=INLINE_GNN_EPOCHS, patience=INLINE_PATIENCE)
    abl[mode] = evaluate_model(m, s, dev)["auroc"]
    print(f"  topology[{mode:7s}] GNN test AUROC={abl[mode]:.4f}")

plt.figure(figsize=(5.5, 4))
plt.bar(list(abl), list(abl.values()), color=["#4C72B0", "#DD8452", "#C44E52"])
plt.axhline(0.5, color="gray", ls="--", lw=1)
plt.ylabel("GNN test AUROC (cold-disease)"); plt.ylim(0.4, 1.0)
plt.title("Topology ablation"); plt.tight_layout(); plt.show()

### 3e. Reproduce the full published benchmark (all 5 seeds + ablations)

The cells above are a fast, one-seed, small-budget illustration. The canonical
numbers in the README come from the full script (5 seeds, real features, XGBoost
tuning). Run it here (uses `--smoke` in `FAST_MODE`):

In [ ]:
SMOKE = "--smoke" if FAST_MODE else ""
!{PY} scripts/run_experiment.py {SMOKE}

In [ ]:
from pathlib import Path
from IPython.display import Image, display
pub = Path("results/oncorepurpose.md")
if pub.exists():
    print("=== Committed full 5-seed published benchmark ===\n")
    print(pub.read_text())
for fig in ["figures/main_results.png", "figures/ablation_topology.png"]:
    if Path(fig).exists():
        display(Image(fig))

## 4. Aim 2 — Mechanism extraction

XGBoost can score a pair but cannot produce a *traceable mechanism*. The graph can.
We build a mechanism adjacency index once, then enumerate
`drug → target → (PPI / pathway) → cancer gene → cancer` chains. Promiscuous carrier
hubs (e.g. albumin) are filtered; remaining bridges are IDF-down-weighted.

In [ ]:
from oncorepurpose.interpret.mechanism_paths import (
    build_mech_index, mechanism_paths, classify_support, mechanism_score,
)
from collections import defaultdict

mech_idx = build_mech_index(data)
rxnames = list(data[DRUG_TYPE].node_names)
dnames  = list(data[DISEASE_TYPE].node_names)

# Map each disease to drugs indicated for it (true indication pairs).
ei = data[target].edge_index
dis2drugs = defaultdict(list)
for dr, ds in zip(ei[0].tolist(), ei[1].tolist()):
    dis2drugs[ds].append(dr)

# Pick a few well-known leukemia/solid-tumour indications and show their MOA paths.
import re
QUERY = r"myeloid leukemia|promyelocytic|colorectal|small cell lung|breast carcinoma|glioblastoma"
shown = 0
for ds, drugs in dis2drugs.items():
    if not (dnames[ds] and re.search(QUERY, str(dnames[ds]), re.I)):
        continue
    dr = drugs[0]
    paths = mechanism_paths(data, mech_idx, dr, ds, max_paths=4)
    if not paths:
        continue
    print(f"\n[{rxnames[dr]} -> {dnames[ds]}]  ({classify_support(paths)}, score={mechanism_score(paths):.2f})")
    for p in paths:
        print(f"   ({p['type']}) {p['text']}")
    shown += 1
    if shown >= 5:
        break

In [ ]:
# Contrast: a random (likely-negative) drug for one of these cancers usually has
# NO specific mechanistic path.
import random
rng = random.Random(0)
ds = next(d for d in dis2drugs if dnames[d] and re.search(QUERY, str(dnames[d]), re.I))
for _ in range(5):
    dr = rng.randrange(len(rxnames))
    if dr in dis2drugs[ds]:
        continue
    paths = mechanism_paths(data, mech_idx, dr, ds, max_paths=3)
    print(f"[{rxnames[dr]} -> {dnames[ds]}]  ({classify_support(paths)})")

## 5. Aim 4 — Mechanism structure separates true vs random pairs

The falsifiable claim, tested **LLM-free** on the graph signal: true oncology
indications carry stronger mechanistic structure than random drug–cancer pairs.
We compute the mechanism score for `N_SEP` true vs `N_SEP` random pairs, then the
**separation AUROC**, direct-target rate, and any-path rate — and plot the
distributions. (Published full run: AUROC **0.879**.)

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from oncorepurpose.interpret.paths import _known_pairs

rng = random.Random(0)
onco_set = set(torch.nonzero(data[DISEASE_TYPE].is_oncology).flatten().tolist())
known = _known_pairs(data)

true_pairs = [(dr, ds) for dr, ds in zip(ei[0].tolist(), ei[1].tolist()) if ds in onco_set]
rng.shuffle(true_pairs); true_pairs = true_pairs[:N_SEP]

num_drugs = int(data[DRUG_TYPE].num_nodes); onco_list = list(onco_set)
neg_pairs, seen = [], set()
while len(neg_pairs) < N_SEP and len(seen) < N_SEP * 40:
    dr = rng.randrange(num_drugs); ds = rng.choice(onco_list)
    if (dr, ds) in known or (dr, ds) in seen:
        continue
    seen.add((dr, ds)); neg_pairs.append((dr, ds))

def score_group(pairs):
    s, direct = [], 0
    for dr, ds in pairs:
        paths = mechanism_paths(data, mech_idx, dr, ds, max_paths=6)
        s.append(mechanism_score(paths))
        if any(p["type"] == "direct_target" for p in paths):
            direct += 1
    return np.array(s), direct / max(1, len(pairs))

s_true, dt_true = score_group(true_pairs)
s_neg,  dt_neg  = score_group(neg_pairs)
y = np.r_[np.ones_like(s_true), np.zeros_like(s_neg)]
auroc = roc_auc_score(y, np.r_[s_true, s_neg])

print(f"Separation AUROC (true vs random): {auroc:.3f}   [n={N_SEP}/group]")
print(f"  mean score      true={s_true.mean():.3f}  random={s_neg.mean():.3f}")
print(f"  direct-target   true={dt_true:.1%}  random={dt_neg:.1%}")
print(f"  any-path        true={(s_true>0).mean():.1%}  random={(s_neg>0).mean():.1%}")

import matplotlib.pyplot as plt
bins = np.linspace(0, max(s_true.max(), 1e-3), 30)
plt.figure(figsize=(7, 4.2))
plt.hist(s_neg,  bins=bins, alpha=0.6, label=f"random (mean {s_neg.mean():.2f})",  color="#9aa0a6")
plt.hist(s_true, bins=bins, alpha=0.6, label=f"true indication (mean {s_true.mean():.2f})", color="#e8684a")
plt.xlabel("graph mechanism score"); plt.ylabel("count")
plt.title(f"Mechanism paths separate true vs random (AUROC {auroc:.2f})")
plt.legend(); plt.tight_layout(); plt.show()

### 5b. Full separation eval + DrugMechDB agreement + hard negatives

The script version runs the full 400/400 separation, a literature-grounding
sample, and the **DrugMechDB** curated-mechanism agreement (UniProt→HGNC mapped;
published agreement **0.802**), then a hard-negative stress test. Both need
network access and degrade gracefully offline.

In [ ]:
!{PY} scripts/evaluate_mechanism.py
!{PY} scripts/evaluate_hard_negatives.py

## 6. Aim 3 — LLM evidence verification

The verifier retrieves Europe PMC **abstracts** for a mechanism path, isolates the
sentences that co-mention the drug and bridge gene, and grades the path
**supported / weak / contradicted / unknown** — via an LLM when a key is set, and a
deterministic lexical-grounding fallback otherwise. Below we run it on one true
indication path and inspect what it retrieved and decided.

In [ ]:
import os
if RUN_LLM and ONCO_LLM_API_KEY:
    os.environ["ONCO_LLM_API_KEY"]  = ONCO_LLM_API_KEY
    os.environ["ONCO_LLM_BASE_URL"] = ONCO_LLM_BASE_URL
    os.environ["ONCO_LLM_MODEL"]    = ONCO_LLM_MODEL

from oncorepurpose.agent.verify import verify_mechanism

# Pick a true indication that has a direct-target path.
chosen = None
for ds, drugs in dis2drugs.items():
    if not (dnames[ds] and re.search(QUERY, str(dnames[ds]), re.I)):
        continue
    paths = mechanism_paths(data, mech_idx, drugs[0], ds, max_paths=3)
    if paths and paths[0]["type"] == "direct_target":
        chosen = paths[0]; break

print("Verifying path:", chosen["text"], "\n")
v = verify_mechanism(chosen, n_lit=4, use_llm=(RUN_LLM and bool(ONCO_LLM_API_KEY)))
print("grade        :", v["grade"], "(source:", v["source"] + ")")
print("abstracts     :", v["n_abstracts"], "| citations:", v["citations"][:3])
print("lexical grade :", v["lexical"]["grade"], "->", v["lexical"]["evidence"])
if v.get("llm"):
    print("LLM grade     :", v["llm"]["grade"], "->", v["llm"].get("evidence", "")[:160])
print("\nTop drug+gene co-mention sentences:")
for s in v["evidence_sentences"][:3]:
    print(f"   [{s['source']}] (moa_cue={s['mechanism_cue']}) {s['sentence'][:160]}")

### 6b. Full LLM verification eval (optional, needs a key)

Published run (`gpt-4o-mini`, 50 true vs 50 random): LLM marks 11 *supported* among
true pairs vs 1 among random; DrugMechDB precision of *supported* is 0.857 (LLM) vs
0.591 (lexical).

In [ ]:
if RUN_LLM and ONCO_LLM_API_KEY:
    !{PY} scripts/verify_llm_eval.py
else:
    print("Skipping live LLM eval (set RUN_LLM=True + a key). Committed summary:")
    import json
    p = Path("results/verify_llm_eval.json")
    if p.exists():
        d = json.loads(p.read_text())
        print("model:", d.get("model"), "| LLM grade dist:", d.get("llm_dist"))
        print("supported rate:", d.get("supported_rate"))
        print("DrugMechDB precision of 'supported':", d.get("dmdb_precision_supported"))

## 7. Finding 4 — Blinded mechanism recovery (the graph's genuine edge)

Train the GNN jointly (link BCE + a contrastive loss that ranks the curated
DrugMechDB bridge gene above degree-matched decoys), then ask it to **name the
bridge gene** for a held-out cold-disease pair. We compare unblinded vs
**mechanism-blinded** (drug→target edge removed). A tabular model has no analogue —
it never embeds a third (gene) node. Here we train on one seed and demonstrate the
ranking for a single held-out pair; the full multi-seed metrics are in the script.

> Needs network access for DrugMechDB; uses real (ST) features.

In [ ]:
from oncorepurpose.evaluation.mechanism_supervision import (
    build_drugmechdb_drug_symbols, build_mech_examples, symbol_to_gene_index,
    DegreeMatchedDecoys,
)
from oncorepurpose.evaluation.trainer import train_gnn_joint
from oncorepurpose.datasets import load_primekg as _load

# Mechanism recovery needs the gene embeddings, so use REAL ST features here.
data_st, targets_st = _load(with_features=True, force_fallback_features=False)
target_st = targets_st["indication"]
in_dims_st = {nt: int(data_st[nt].x.size(1)) for nt in data_st.node_types}
GENE = "gene_protein"

dmdb = build_drugmechdb_drug_symbols()
sym2gidx = symbol_to_gene_index(data_st)
print(f"DrugMechDB drugs mapped: {len(dmdb)}")

set_all_seeds(SEED)
sp = make_split(data_st, target_st, "inductive_cold_dst", seed=SEED, restrict_oncology=True)

def positives(eli, lab):
    return eli[:, lab.bool()]
mech_tr = build_mech_examples(data_st, positives(sp.train_label_index, sp.train_label), dmdb, sym2gidx)
mech_te = build_mech_examples(data_st, positives(sp.test_label_index,  sp.test_label),  dmdb, sym2gidx)
print(f"covered train pairs={len(mech_tr.pairs)}  test pairs={len(mech_te.pairs)}")

In [ ]:
from oncorepurpose.interpret.mechanism_paths import build_mech_index as _bmi
mech_idx_st = _bmi(data_st)
num_genes = int(data_st[GENE].num_nodes)
decoys = DegreeMatchedDecoys(mech_idx_st["prot_drug_deg"], num_genes, seed=SEED)

# Joint model (link + mechanism aux loss) vs link-only (same head, no aux loss).
set_all_seeds(SEED)
joint = HeteroGNN(list(data_st.node_types), list(sp.base.edge_types), in_dims_st).to(dev)
joint = train_gnn_joint(joint, sp, dev, mech_tr, decoys, lam=1.0, epochs=INLINE_REC_EPOCHS)
set_all_seeds(SEED)
linkonly = HeteroGNN(list(data_st.node_types), list(sp.base.edge_types), in_dims_st).to(dev)
linkonly = train_gnn(linkonly, sp, dev, epochs=INLINE_REC_EPOCHS)
print("trained joint + link-only GNNs")

In [ ]:
# Rank all genes as the bridge for one held-out (drug, cancer) pair.
gnames = list(data_st[GENE].node_names)
di, ci, true_genes = mech_te.pairs[0]
print(f"Held-out pair: {rxnames[di]} -> {dnames[ci]}")
print("Curated DrugMechDB bridge gene(s):", [gnames[g] for g in true_genes], "\n")

@torch.no_grad()
def rank_bridge(model, base):
    z = model.encode(base)
    g = torch.arange(num_genes)
    d = torch.full((num_genes,), di); c = torch.full((num_genes,), ci)
    scores = model.score_mechanism(z, d, g, c).cpu()
    order = torch.argsort(scores, descending=True).tolist()
    return order

order_joint = rank_bridge(joint, sp.base)
order_link  = rank_bridge(linkonly, sp.base)
def rank_of(order):
    pos = {g: i for i, g in enumerate(order)}
    return min((pos[g] for g in true_genes if g in pos), default=None)
print("Top-10 genes ranked by JOINT GNN:", [gnames[g] for g in order_joint[:10]])
print(f"  true bridge gene rank (joint)     : {rank_of(order_joint)}")
print(f"  true bridge gene rank (link-only) : {rank_of(order_link)}")

### 7b. Full multi-seed mechanism-recovery metrics

In [ ]:
!{PY} scripts/evaluate_mechanism_recovery.py {SMOKE}
import json
p = Path("results/mechanism_recovery_eval.json")
if p.exists():
    print("\n" + json.loads(p.read_text()).get("interpretation", ""))

## 8. Deliverable — vetted oncology repurposing shortlist

Train a deployment GNN on all indication edges, rank **novel** drug candidates for
chosen cancers (by disease-specific lift, excluding known indications), and attach
the multi-hop KG rationale for each. We build the shortlist inline as a table.

In [ ]:
from oncorepurpose.interpret.paths import predict_candidates_for_diseases, extract_paths

# Train a deployment GNN on the full indication graph (transductive, no test holdout).
set_all_seeds(SEED)
dep_split = make_split(data, target, "transductive", seed=SEED, val_frac=0.1, test_frac=0.0)
dep_gnn = HeteroGNN(list(data.node_types), list(dep_split.base.edge_types), in_dims,
                    hidden=128, num_layers=2, dropout=0.3)
dep_gnn = train_gnn(dep_gnn, dep_split, dev, epochs=INLINE_GNN_EPOCHS, patience=INLINE_PATIENCE)

# Pick a couple of well-connected oncology diseases by name.
def pick_disease(substr):
    deg = defaultdict(int)
    for d in ei[1].tolist():
        deg[d] += 1
    cands = [i for i, n in enumerate(dnames)
             if n and substr.lower() in str(n).lower() and i in onco_set]
    return max(cands, key=lambda i: deg[i]) if cands else None

disease_idx = [d for d in (pick_disease("glioblastoma"), pick_disease("pancreatic")) if d is not None]
print("Diseases:", [dnames[d] for d in disease_idx])
preds = predict_candidates_for_diseases(dep_gnn, data, target, disease_idx, dev,
                                        top_k=5, exclude_known=True)

In [ ]:
import pandas as pd
rows = []
for dz in disease_idx:
    for drug_i, score, lift in preds[dz]:
        paths = extract_paths(data, drug_i, dz, max_paths=2)
        rows.append({
            "disease": dnames[dz],
            "candidate_drug": rxnames[drug_i],
            "model_score": round(score, 3),
            "specificity_lift": round(lift, 3),
            "top_KG_rationale": paths[0]["text"] if paths else "(no 2-hop bridge found)",
        })
display(pd.DataFrame(rows))

### 8b. Full deliverable via the script (literature + optional LLM dossier)

In [ ]:
if FAST_MODE:
    REPORT_ARGS = '--diseases glioblastoma "pancreatic cancer" --top-k 3 --gnn-epochs 5 --fallback-features --no-llm'
else:
    REPORT_ARGS = '--diseases glioblastoma "pancreatic cancer" --top-k 5'
    if not (RUN_LLM and ONCO_LLM_API_KEY):
        REPORT_ARGS += " --no-llm"
!{PY} scripts/generate_report.py {REPORT_ARGS}
p = Path("results/repurposing_shortlist.md")
if p.exists():
    t = p.read_text()
    print(t[:3000] + ("\n... (truncated)" if len(t) > 3000 else ""))

## Wrap-up

You've now run OncoEvidence end to end — and seen the actual code at every stage,
not just script output.

**To reproduce the published numbers**, set `FAST_MODE = False` (5 seeds, real
SentenceTransformer features, XGBoost tuning) and, with a key, `RUN_LLM = True`.

**Findings**
1. On link ranking alone, a tuned XGBoost matches/beats the GNN — the graph isn't
   necessary for that task.
2. But the graph carries traceable mechanism a tabular model cannot produce.
3. Mechanism structure separates true vs random oncology pairs (AUROC 0.879).
4. The graph's genuine edge: blinded mechanism recovery (R@10 0.25 where
   trivial/link-only baselines hit 0).

> *All predictions are hypothesis-generating and not medical advice.*